# Practical Exam: Sample SQL Associate

Tech Solutions Inc. is a leading technology company specializing in software development and IT consulting services. The company prides itself on delivering innovative solutions to clients across various industries. With a dedicated team of skilled professionals, TechSolutions has earned a reputation for excellence in the tech industry.

Tech Solutions Inc. has been experiencing a decline in customer satisfaction ratings over the past few months. Customer feedback surveys and support tickets indicate an increase in dissatisfaction among clients. The company is concerned about this trend as it directly impacts customer retention, reputation, and overall business growth.

You are working with the customer support team to provide data to managers to help the company take proactive measures to address these concerns effectively. 

## Data

The following schema diagram shows the tables available.

![customer_feedback](support_db.png)

# Task 1

Before you can start any analysis, you need to confirm that the data is accurate and reflects what you expect to see. 

It is known that there are some issues with the `support` table, and the data team have provided the following data description. 

Write a query to return data matching this description. You must match all column names and description criteria.

| Column Name | Criteria                                                |
|-------------|---------------------------------------------------------|
|id | Discrete. The unique identifier of the support ticket. </br>Missing values are not possible due to the database structure.|
| customer_id | Discrete. The unique identifier of the customer. </br>Missing values should be replaced with 0.|
| category | Nominal. The category of the support request, can be one of Feedback, Billing Enquiry, Bug, Installation Problem, Other. </br>Missing values should be replaced with Other. |
| status | Nominal. The current status of the support ticket, one of Open, In Progress or Resolved. </br>Missing values should be replaced with 'Resolved'. |
| creation_date | Discrete. The date the ticket was created. Can be any date in 2023. </br>Missing values should be replaced with 2023-01-01. |
| response_time | Discrete. The number of days taken to respond to the support ticket. </br>Missing values should be replaced with 0. |
| resolution_time | Continuous. The number of hours taken to resolve the support ticket, rounded to 2 decimal places. </br>Missing values should be replaced with 0. |

In [ ]:
#Task 1 Query

#To get data types of columns
SELECT *
FROM INFORMATION_SCHEMA.columns
WHERE table_name = 'support';

#To investigate value counts of some column
SELECT status, COUNT(status)
FROM support
GROUP BY status
ORDER BY status;

#Columns to fix: category (typo), status (missing values), resolution_time (values)
#Columns that are good: id, customer_id, creation_date, response_time

#Adjusting columns as necessary
    #Column 'category': Values "Billing enquiry" need to be changed to "Billing Enquiry"
SELECT INITCAP(category) ...;
    #Column 'status': Are some missing values ("-") that need to be replace with "Resolved"
SELECT REPLACE(status, '-', 'Resolved') ...;
    #Column 'resolution_time': Each value includes " hours". Want to get rid of this part to include a numerical value 
        #(to 2 decimals). Also, change data type from text to float.
SELECT REPLACE(resolution_time, ' hours', '') ...;

#Check column adjustments
WITH task1_adj AS (
    SELECT id, customer_id, 
        INITCAP(category) AS category, 
        REPLACE(status, '-', 'Resolved') AS status, 
        creation_date, response_time, 
        REPLACE(resolution_time, ' hours', '')::FLOAT(2) AS resolution_time
     FROM support)

SELECT *
FROM task1_adj;

#Output final query
SELECT id, customer_id,
    INITCAP(category) AS category,
    REPLACE(status, '-', 'Resolved') AS status,
    creation_date, response_time,
    REPLACE(resolution_time, ' hours', '')::FLOAT(2) AS resolution_time
FROM support;
    #returns 1,987 rows

# Task 2

It is suspected that the response time to tickets is a big factor in unhappiness.

Calculate the minimum and maximum response time for each category of support ticket. 

Your output should include the columns `category`, `min_response` and `max_response`. 

Values should be rounded to two decimal places where appropriate. 

In [ ]:
#Task 2 Query

SELECT category, MIN(response_time) AS min_response, MAX(response_time) AS max_response
FROM support
GROUP BY category;
    #returns 5 rows

# Task 3

The support team want to know more about the `rating` provided by customers who reported `Bugs` or `Installation Problem`s. 

Write a query to return the `rating` from the survey, the `customer_id`, `category` and `response_time` of the support ticket, for the customers & categories of interest. 

Use the original support table, not the output of task 1. 

In [ ]:
#Task 3 Query
    #are 1,987 rows in the "support" table, 200 rows in the "survey" table
    #are 128 total common data points across the two tables because there are several skipped numbers in the 'id' column
        #of the "support" table (spans 1-3,000 even though there are only 1,987 rows)

SELECT rating, support.customer_id, category, response_time
FROM survey
LEFT JOIN support
ON survey.customer_id = support.customer_id
WHERE category IN ('Bug','Installation Problem')
ORDER BY support.customer_id;
    #returns 123 rows